In [9]:
import pandas as pd
import numpy as  np
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
import matplotlib.pyplot as plt
import datetime as dt
from calendar import day_name

In [10]:
user = "wconceicao"
password = "zJm7$j%qRU@WoCxM"
host = "dw-ro.data.grupoa.education"
port = "5432"
database = "postgres"

password_encoded = quote_plus(password)

engine = create_engine(
    f"postgresql+psycopg2://{user}:{password_encoded}@{host}:{port}/{database}"
)

start_date = '2025-09-01'
end_date = '2025-09-09'
campaigns = (1605, 1553, 1025, 1700)


query = text(f"""

WITH ranked_calls AS (
    SELECT
        RIGHT(phone_number,11) AS phone,

        ROW_NUMBER() OVER (
            PARTITION BY RIGHT(phone_number,11)
            ORDER BY start_agent_date DESC
        ) AS rn,
        tablename,
        CASE
            WHEN tablename ~* 'mental|psicologia' THEN 'Psicologia'
            WHEN tablename ~* 'multi' THEN 'Multi'
            WHEN tablename ~* 'fisio' THEN 'Fisioterapia'
            WHEN tablename ~* 'enf' THEN 'Enfermagem'
            WHEN tablename ~* 'medic' THEN 'Medicina'
            WHEN tablename ~* 'nutri' THEN 'Nutrição'
            WHEN tablename ~* 'vet' THEN 'Veterinária'
            WHEN tablename ~* 'ped' THEN 'Pediatria'
            WHEN tablename ~* 'psiquia' THEN 'Psiquiatria'
            ELSE 'Outras Áreas'
        END AS area,

        CASE
            WHEN campaign = '1553' THEN 'receptivoWay'
            WHEN tablename ~* 'legado' THEN 'legado'
            WHEN tablename ~* 'evento' THEN 'evento'
            WHEN tablename ~* 'cancel' THEN 'cancelados'
            WHEN tablename ~* 'INATIV' THEN 'inativa'
            WHEN tablename ~* 'ATIV' THEN 'ativa'
            WHEN tablename ~* 'MATERIAL' THEN 'Material Rico'
            WHEN tablename ~* 'leads' THEN 'Base Leads' 
            WHEN tablename ~* 'carri' THEN 'Carrinho'  
            ELSE NULL
        END AS base_type,
        EXTRACT(HOUR FROM start_agent_date) AS hour,


        CASE
            WHEN EXTRACT(EPOCH FROM wrap_duration) > 0 THEN 1
            ELSE 0
        END AS atendida,

        DATE(start_agent_date) AS data

    FROM integration_operations.vw_call_center_calls
    WHERE campaign_id IN (1025,1553,1605,1690,1299)
    AND start_agent_date::date >= current_date - interval '5 months'
)

SELECT
    area,
    base_type,
    tablename,
    data,
    hour,
    COUNT(*) AS tentativas,
    SUM(atendida) AS atendidas
FROM ranked_calls
GROUP BY area, data,base_type,hour, tablename
    
""")

try:
    df = pd.read_sql(query, engine)
    num_linhas = df.shape[0]
    print(f"Consulta executada com sucesso! Retornou {num_linhas} linhas.")
    hoje = dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"Data e hora da execução: {hoje}")
except Exception as e:
    df = pd.read_excel(r'C:\Users\wconceicao\OneDrive - Grupo A Educação SA\Área de Trabalho\Construtores\backup.xlsx')
    
    print("❌ Erro ao executar a consulta:")
    print(f"➡️ {e}")
    print(f'Lendo arquivo auxiliar...')

Consulta executada com sucesso! Retornou 9525 linhas.
Data e hora da execução: 2026-04-07 11:45:09


In [11]:
df['data'] = pd.to_datetime(df['data'])

df['dia_semana'] = df['data'].dt.day_name()
day_name = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

traduzir_dia = {
    'Monday': 'Segunda',
    'Tuesday': 'Terça',
    'Wednesday': 'Quarta',  
    'Thursday': 'Quinta',
    'Friday': 'Sexta',
    'Saturday': 'Sábado',
    'Sunday': 'Domingo'
}

df['dia_semana'] = df['dia_semana'].map(traduzir_dia)

df['week_month'] = ((df['data'].dt.day - 1) // 7) + 1
df['day_week'] = df['dia_semana'] + '_W' + df['week_month'].astype(str)


In [12]:
hora_loc = df.copy()
hora_loc = hora_loc[hora_loc['day_week'] == 'Quarta_W4']
hora_loc = hora_loc.fillna(0)
hora_loc['tx_loc'] = (hora_loc['atendidas'] / hora_loc['tentativas'] * 100).round(2)
hora_loc = hora_loc.groupby(['hour','area']).agg(
    loc = ('tx_loc','mean')
).reset_index()
hora_loc['loc'] = hora_loc['loc'].round(2)
hora_loc = hora_loc.fillna(0)
pivot_loc = pd.pivot_table(
    hora_loc,
    values = 'loc',
    columns='area',
    index='hour'
)
pivot_loc.style.background_gradient(cmap='Blues')

area,Enfermagem,Fisioterapia,Medicina,Multi,Outras Áreas,Pediatria,Psicologia,Veterinária
hour,,,,,,,,
9.000000,2.350000,2.470000,1.390000,2.860000,51.500000,nan,3.820000,2.780000
10.000000,2.190000,2.440000,3.270000,2.780000,49.810000,nan,3.260000,3.670000
11.000000,1.520000,2.940000,2.290000,2.650000,43.560000,3.120000,1.450000,3.820000
12.000000,1.540000,2.920000,2.540000,2.020000,58.360000,4.050000,2.500000,2.380000
13.000000,0.000000,2.880000,2.880000,nan,16.470000,5.680000,2.500000,1.840000
14.000000,3.080000,2.420000,1.980000,nan,39.190000,4.890000,3.230000,2.820000
15.000000,0.000000,1.950000,2.410000,nan,41.410000,3.840000,2.070000,2.380000
16.000000,0.000000,3.770000,2.720000,nan,40.740000,3.580000,2.140000,1.190000
17.000000,0.000000,2.570000,1.740000,nan,33.330000,2.740000,0.000000,7.140000


In [13]:
top_loc = df.copy()
top_loc = top_loc.groupby(['area','base_type']).agg(
    media_tent = ('tentativas','mean'),
    media_atend = ('atendidas','mean')
).reset_index()
top_loc['tx_loc'] = ((top_loc['media_atend'] / top_loc['media_tent']) * 100).round(2)
top_loc['media_atend'] = top_loc['media_atend'].round(0)
top_loc['media_tent'] = top_loc['media_tent'].round(0)
top_loc = top_loc.sort_values(by='tx_loc',ascending=False)
top_loc = top_loc.loc[
    (top_loc['media_tent'] >= 100) & 
    (top_loc['tx_loc'] >= 2.0)
]
top_loc.head(10)

,area,base_type,media_tent,media_atend,tx_loc
27,Multi,evento,118.0,5.0,4.29
25,Multi,Material Rico,237.0,10.0,4.19
39,Pediatria,evento,283.0,11.0,3.93
5,Enfermagem,evento,211.0,8.0,3.65
28,Multi,inativa,176.0,6.0,3.60
34,Outras Áreas,evento,164.0,6.0,3.40
7,Enfermagem,legado,341.0,11.0,3.37
8,Fisioterapia,Base Leads,220.0,7.0,3.33
32,Nutrição,evento,181.0,6.0,3.31
0,Enfermagem,Base Leads,258.0,8.0,3.29


In [14]:
hoje = dt.date.today()  

# garante que coluna data é datetime
df['data'] = pd.to_datetime(df['data'])

# filtra para hoje
localizacao_hoje = df.loc[df['data'].dt.date == hoje].copy()

# agrupa
localizacao_hoje = localizacao_hoje.groupby(['area','base_type']).agg(
    tentativas=('tentativas','sum'),
    atendidas=('atendidas','sum')
).reset_index()

localizacao_hoje['tx_loc'] = (localizacao_hoje['atendidas'] / localizacao_hoje['tentativas'] * 100).round(2)

localizacao_hoje = localizacao_hoje.sort_values(by='tx_loc', ascending=False)

localizacao_hoje

,area,base_type,tentativas,atendidas,tx_loc
0,Enfermagem,Base Leads,290,12,4.14
2,Fisioterapia,Base Leads,37,1,2.70
1,Enfermagem,inativa,115,2,1.74
3,Fisioterapia,inativa,580,7,1.21
5,Nutrição,inativa,294,1,0.34
6,Psiquiatria,ativa,581,2,0.34
4,Medicina,Base Leads,3,0,0.00


In [17]:
historico_mes = df.copy()
historico_mes['data'] = pd.to_datetime(df['data'])

historico_mes = historico_mes.groupby(['data','area','base_type']).agg(
    tentativas = ('tentativas','sum'),
    atendidas = ('atendidas','sum')
).reset_index()
historico_mes = historico_mes[historico_mes['data'] == '2026-04-07']
historico_mes = historico_mes.sort_values(by='data',ascending=True)
historico_mes

,data,area,base_type,tentativas,atendidas
904,2026-04-07,Enfermagem,Base Leads,290,12
905,2026-04-07,Enfermagem,inativa,115,2
906,2026-04-07,Fisioterapia,Base Leads,37,1
907,2026-04-07,Fisioterapia,inativa,580,7
908,2026-04-07,Medicina,Base Leads,3,0
909,2026-04-07,Nutrição,inativa,294,1
910,2026-04-07,Psiquiatria,ativa,581,2


In [16]:
localizacao = df.copy()
localizacao = localizacao.groupby(['data','area','base_type','tablename']).agg(
    tentativas = ('tentativas','sum'),
    atendidas = ('atendidas','sum')
).reset_index()
localizacao['tipo_discagem'] = np.where(localizacao['data'] <= '2026-03-11','Automatica','Preditivo')